# S6E5 — Ensemble

## 1. Imports e configuração

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

from mltemplate.config import ProjectConfig
from mltemplate.storage import StorageManager
from mltemplate.data import KaggleSource, DataManager

import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
config = ProjectConfig(
    target="PitNextLap",
    numerical_features=[],    # manter igual aos notebooks anteriores
    categorical_features=[],  # manter igual aos notebooks anteriores
    ignore_features=["id"],
    problem_type="regression",
)

storage = StorageManager(root=Path("."))
dm      = DataManager(storage, config)

## 2. Carregar modelos e feature set

In [ ]:
xgb_model  = storage.load_model("xgb_v1")
lgbm_model = storage.load_model("lgbm_v1")
models = [xgb_model, lgbm_model]

X_train_fe, X_test_fe = dm.load_feature_set(name="v1")

source = KaggleSource("playground-series-s6e5")
train_df, test_df = dm.load_raw(source)
X_train, X_val, y_train, y_val = dm.split(train_df)

print(f"X_train: {X_train_fe.shape}")
print(f"X_test:  {X_test_fe.shape}")

## 3. Pesos do ensemble

> `EnsembleCreator` suporta apenas métricas de classificação (`roc_auc`, `accuracy`). Para regressão, usamos média ponderada manual — ajuste os pesos com base nas métricas de validação de `03_tuning.ipynb`.

In [ ]:
# Ajuste os pesos conforme os RMSE obtidos no tuning (peso maior = modelo melhor)
weights = np.array([0.5, 0.5])
weights = weights / weights.sum()

print(f"Pesos: XGB={weights[0]:.2f}  LGBM={weights[1]:.2f}")

## 4. Avaliar no conjunto de validação

In [ ]:
# Aplicar o mesmo pipeline de features ao X_val antes de avaliar
# (copiar o bloco de transformação do 02_features.ipynb)

# preds_xgb_val  = xgb_model.predict(X_val_fe)
# preds_lgbm_val = lgbm_model.predict(X_val_fe)
# preds_val      = weights[0] * preds_xgb_val + weights[1] * preds_lgbm_val

# rmse = mean_squared_error(y_val, preds_val, squared=False)
# print(f"Ensemble RMSE (val): {rmse:.4f}")

## 5. Submissão

In [ ]:
preds_xgb  = xgb_model.predict(X_test_fe)
preds_lgbm = lgbm_model.predict(X_test_fe)
preds_final = weights[0] * preds_xgb + weights[1] * preds_lgbm

submission = pd.DataFrame({"id": test_df["id"], config.target: preds_final})
storage.save_submission(submission, "ensemble_v1")

print(f"Submissão salva em: {storage.submissions_path / 'ensemble_v1.csv'}")
submission.head()